# Top Tagging with PFN + ArcFace (Google Drive Version)
This notebook implements a PFN with an ArcFace Prototype Layer using the dataset stored in your Google Drive.

**Path:** `MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/`

In [1]:
# 1. Install Dependencies
%pip install -q awkward fastjet h5py hdf5plugin vector scikit-learn matplotlib seaborn

In [2]:
# 2. Mount Google Drive and Set Paths
from google.colab import drive
import os
drive.mount('/content/drive')

# Set your data folder here
DRIVE_PATH = '/content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/'
train_h5 = os.path.join(DRIVE_PATH, 'train.h5')
val_h5 = os.path.join(DRIVE_PATH, 'val.h5')
test_h5 = os.path.join(DRIVE_PATH, 'test.h5')

for f in [train_h5, val_h5, test_h5]:
    if not os.path.exists(f):
        print(f"Warning: {f} not found!")
    else:
        print(f"Found: {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: /content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/train.h5
Found: /content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/val.h5
Found: /content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/test.h5


In [3]:
import json
import time
import numpy as np
import awkward as ak
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import h5py
import hdf5plugin
import vector
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, precision_recall_curve, average_precision_score
import fastjet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 3. Data Loading & Utilities

In [4]:
def load_awkward(h5_path, max_events=None, trim_padding=True):
    print(f"Reading {h5_path}...")
    with h5py.File(h5_path, 'r') as f:
        # 1. Locate dataset
        if 'table' in f:
            obj = f['table']
            dataset = obj['table'] if isinstance(obj, h5py.Group) else obj
        else:
            dataset = f[list(f.keys())[0]]

        n = min(dataset.shape[0], max_events) if max_events else dataset.shape[0]
        raw_data = dataset[:n]

    # 2. Handle structured array (values_block_0 / values_block_1 layout)
    if raw_data.dtype.names is not None:
        if 'values_block_0' in raw_data.dtype.names:
            # Known Top-Tagging layout:
            #   values_block_0: (N, 804) — 200 particles × 4 (E,PX,PY,PZ) interleaved + 4 truth cols
            #   values_block_1: (N, 2)   — col0=ttv split, col1=is_signal_new (label)
            v0 = raw_data['values_block_0'].astype(np.float32)  # (N, 804)
            v1 = raw_data['values_block_1'].astype(np.float32)  # (N, 2)
            N_CONS = 200
            es  = v0[:, 0:N_CONS*4:4]   # (N, 200) — E
            pxs = v0[:, 1:N_CONS*4:4]   # (N, 200) — PX
            pys = v0[:, 2:N_CONS*4:4]   # (N, 200) — PY
            pzs = v0[:, 3:N_CONS*4:4]   # (N, 200) — PZ
            p4_data = np.stack([es, pxs, pys, pzs], axis=2)  # (N, 200, 4)
            labels  = v1[:, 1].astype(np.int8)                # is_signal_new
            weights = np.ones(n, dtype=np.float32)
        else:
            # Generic structured array: stack all numeric fields column-wise
            # (only safe if every field has the same shape)
            fields = [nm for nm in raw_data.dtype.names if nm != 'index']
            cols = [raw_data[f].astype(np.float32) for f in fields]
            # If fields are 1-D per event, stack → (N, F); else fall through
            stacked = np.column_stack(cols)           # (N, F)
            labels  = stacked[:, 800].astype(np.int8) if stacked.shape[1] > 800 else np.zeros(n, dtype=np.int8)
            p4_data = stacked[:, :800].reshape(-1, 200, 4)
            weights = stacked[:, 801] if stacked.shape[1] > 801 else np.ones(n, dtype=np.float32)
    else:
        # Plain float array — standard (N, 802) layout
        raw = raw_data.astype(np.float32)
        p4_data = raw[:, :800].reshape(-1, 200, 4)
        labels  = raw[:, 800].astype(np.int8) if raw.shape[1] > 800 else np.zeros(n, dtype=np.int8)
        weights = raw[:, 801] if raw.shape[1] > 801 else np.ones(n, dtype=np.float32)

    # 3. Build Awkward jagged array (trim zero-padded constituents)
    if trim_padding:
        mask   = p4_data[:, :, 0] > 0            # E > 0 → real constituent
        counts = mask.sum(axis=1)
        flat   = p4_data[mask]                    # (total_constituents, 4)
        events = ak.unflatten(
            ak.zip({"E": flat[:,0], "px": flat[:,1], "py": flat[:,2], "pz": flat[:,3]},
                   with_name="Momentum4D"),
            counts
        )
        print(f"  {n:,} events, {int(counts.sum()):,} constituents ({counts.mean():.1f} avg)")
    else:
        events = ak.zip(
            {"E": p4_data[:,:,0], "px": p4_data[:,:,1], "py": p4_data[:,:,2], "pz": p4_data[:,:,3]},
            with_name="Momentum4D"
        )

    return events, labels, weights


def compute_features(events):
    jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, 0.8)
    cluster = fastjet.ClusterSequence(events, jet_def)
    jets = cluster.inclusive_jets(min_pt=200)
    jets = jets[ak.argsort(jets.pt, axis=-1, ascending=False)[:, :1]]
    mass = ak.to_numpy(ak.flatten(jets.mass))
    pt   = ak.to_numpy(ak.flatten(jets.pt))
    return np.stack([mass, pt], axis=1)


## 4. Model Architecture

In [5]:
def get_relative_features(constituents, mask=None):
    E = constituents[:, :, 0]; px = constituents[:, :, 1]; py = constituents[:, :, 2]; pz = constituents[:, :, 3]
    pt = torch.sqrt(px**2 + py**2 + 1e-10)
    eta = torch.asinh(pz / (pt + 1e-10)); phi = torch.atan2(py, px)
    jet_p4 = constituents.sum(dim=1)
    jet_pt = torch.sqrt(jet_p4[:, 1]**2 + jet_p4[:, 2]**2 + 1e-10)
    jet_eta = torch.asinh(jet_p4[:, 3] / (jet_pt + 1e-10)); jet_phi = torch.atan2(jet_p4[:, 2], jet_p4[:, 1])
    d_y = eta - jet_eta.unsqueeze(1)
    d_phi = (phi - jet_phi.unsqueeze(1) + np.pi) % (2 * np.pi) - np.pi
    features = torch.stack([torch.log(pt + 1e-10), torch.log(E + 1e-10), d_y, d_phi, pt/jet_pt.unsqueeze(1)], dim=2)
    if mask is not None: features = features * mask.unsqueeze(-1).float()
    return features

class PFN(nn.Module):
    def __init__(self, input_dim=5, particle_dim=256, jet_dim=256):
        super().__init__()
        self.phi = nn.Sequential(nn.Linear(input_dim, particle_dim), nn.BatchNorm1d(particle_dim), nn.ReLU(), nn.Linear(particle_dim, particle_dim), nn.BatchNorm1d(particle_dim), nn.ReLU())
        self.rho = nn.Sequential(nn.Linear(particle_dim, jet_dim), nn.BatchNorm1d(jet_dim), nn.ReLU(), nn.Linear(jet_dim, jet_dim), nn.BatchNorm1d(jet_dim), nn.ReLU(), nn.Linear(jet_dim, jet_dim))
    def forward(self, x, mask=None):
        b, n, d = x.shape
        p = self.phi(x.reshape(b*n, d)).reshape(b, n, -1)
        p = p.sum(dim=1) if mask is not None else p.mean(dim=1)
        return self.rho(p)

class ArcFacePrototypeLayer(nn.Module):
    def __init__(self, input_dim, n_prototypes=20, s=32.0, m=0.5):
        super().__init__()
        self.s = s; self.m = m
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, input_dim))
        nn.init.xavier_uniform_(self.prototypes)
    def forward(self, x, labels=None):
        x_n = F.normalize(x, dim=1); p_n = F.normalize(self.prototypes, dim=1)
        sims = x_n @ p_n.T
        logits = torch.stack([sims[:, :10].max(1)[0], sims[:, 10:].max(1)[0]], dim=1)
        if self.training and labels is not None:
            theta = torch.acos(logits.clamp(-1+1e-7, 1-1e-7))
            logits = torch.where(F.one_hot(labels, 2).bool(), torch.cos(theta + self.m), logits)
        return logits * self.s, sims

class TopTaggingModel(nn.Module):
    def __init__(self): super().__init__(); self.pfn = PFN(); self.proto = ArcFacePrototypeLayer(256)
    def forward(self, x, labels=None, mask=None):
        emb = self.pfn(get_relative_features(x, mask), mask)
        return self.proto(emb, labels)

## 5. Training

In [12]:
class JetDataset(Dataset):
    def __init__(self, h5, max_events=None):
        self.ev, lab, _ = load_awkward(h5, max_events)
        self.n   = len(self.ev)
        self.lab = torch.from_numpy(np.array(lab)).long()
        self.c   = np.zeros((self.n, 200, 4), dtype=np.float32)
        self.m   = np.zeros((self.n, 200),    dtype=bool)
        for i in range(self.n):
            np_ = min(len(self.ev[i]), 200)
            self.c[i, :np_, 0] = self.ev[i].E[:np_]
            self.c[i, :np_, 1] = self.ev[i].px[:np_]
            self.c[i, :np_, 2] = self.ev[i].py[:np_]
            self.c[i, :np_, 3] = self.ev[i].pz[:np_]
            self.m[i, :np_]    = True

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return self.c[idx], self.lab[idx], self.m[idx]


def run_train():
    train_ds = JetDataset(train_h5, max_events=50000)
    val_ds   = JetDataset(val_h5,   max_events=50000)
    loader   = DataLoader(train_ds, batch_size=1024, shuffle=True,  num_workers=4, pin_memory=True)
    v_loader = DataLoader(val_ds,   batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)

    model = TopTaggingModel().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)

    best_acc = 0.0
    for ep in range(30):
        # --- Train ---
        model.train()
        train_correct, train_total = 0, 0
        for x, y, m in loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            opt.zero_grad()
            logits, _ = model(x, labels=y, mask=m)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total   += y.size(0)
        train_acc = train_correct / train_total

        # --- Validate ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for x, y, m in v_loader:
                x, y, m = x.to(device), y.to(device), m.to(device)
                logits, _ = model(x, mask=m)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total   += y.size(0)
        val_acc = val_correct / val_total

        print(f"Epoch {ep+1:02d}: Train Acc = {train_acc:.4f} | Val Acc = {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pt')
            print(f"  ✓ Saved best model (val_acc={best_acc:.4f})")

run_train()


Reading /content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/train.h5...
  50,000 events, 2,452,588 constituents (49.1 avg)
Reading /content/drive/MyDrive/Jet Tagging/Top Quark Tagging Reference Dataset/val.h5...
  50,000 events, 2,458,438 constituents (49.2 avg)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 01: Train Acc = 0.6713 | Val Acc = 0.8340
  ✓ Saved best model (val_acc=0.8340)
Epoch 02: Train Acc = 0.8172 | Val Acc = 0.7618
Epoch 03: Train Acc = 0.8165 | Val Acc = 0.8701
  ✓ Saved best model (val_acc=0.8701)
Epoch 04: Train Acc = 0.8242 | Val Acc = 0.8754
  ✓ Saved best model (val_acc=0.8754)
Epoch 05: Train Acc = 0.8308 | Val Acc = 0.8542
Epoch 06: Train Acc = 0.8316 | Val Acc = 0.8421
Epoch 07: Train Acc = 0.8345 | Val Acc = 0.8866
  ✓ Saved best model (val_acc=0.8866)
Epoch 08: Train Acc = 0.8428 | Val Acc = 0.6965
Epoch 09: Train Acc = 0.8418 | Val Acc = 0.8892
  ✓ Saved best model (val_acc=0.8892)
Epoch 10: Train Acc = 0.8459 | Val Acc = 0.5407
Epoch 11: Train Acc = 0.8429 | Val Acc = 0.7303
Epoch 12: Train Acc = 0.8423 | Val Acc = 0.8604
Epoch 13: Train Acc = 0.8433 | Val Acc = 0.8898
  ✓ Saved best model (val_acc=0.8898)
Epoch 14: Train Acc = 0.8449 | Val Acc = 0.6600
Epoch 15: Train Acc = 0.8424 | Val Acc = 0.8642
Epoch 16: Train Acc = 0.8458 | Val Acc = 0.8368
Epoc